# 05c — Feature Engineering V3: Inbound Delays + Tail History + Hub Fever

Adds real-time airport state features: inbound arrival delays, previous tail delay, and national hub congestion.

**Input:** `dataset/merged_flights_fe.parquet`

**Output:** `dataset/merged_flights_fe.parquet` (updated)

**Features added:** inbound_arr_delay_3h, dest_inbound_arr_delay_3h, prev_tail_arr_delay, national_hub_delay_2h, tail_flight_num_today, dest_hourly_flight_count, airline_delay_rate_7d, origin_delay_rate_7d

In [1]:
import pandas as pd

In [2]:
flights=pd.read_parquet("/Users/harshithnr/flight_delay_predictions/dataset/merged_flights_fe.parquet")

## Tail Activity Features
tail_flight_num_today: nth flight of the day for this aircraft. Higher values correlate with accumulated delay.

In [ ]:
flights_sorted = flights.sort_values(['TAIL_NUM', 'DEP_DATETIME'])
flights['tail_flight_num_today'] = flights_sorted.groupby(['TAIL_NUM', 'FL_DATE']).cumcount().astype('int8')
print("=== tail_flight_num_today ===")
print(flights['tail_flight_num_today'].value_counts().sort_index().head(15))
print(f"\nDelay rate by flight number today:")
print(flights.groupby('tail_flight_num_today')['ARR_DEL15'].mean().round(4).head(12) * 100)

dest_hourly = flights.groupby(['DEST', 'FL_DATE', 'ARR_HOUR']).size().reset_index(name='dest_hourly_flight_count')
flights = flights.merge(dest_hourly, on=['DEST', 'FL_DATE', 'ARR_HOUR'], how='left')
print(f"\n=== dest_hourly_flight_count ===")
print(f"Nulls: {flights['dest_hourly_flight_count'].isna().sum()}")
print(flights['dest_hourly_flight_count'].describe().round(2))

flights['dep_minutes_since_midnight'] = (
    flights['CRS_DEP_TIME'] // 100 * 60 + flights['CRS_DEP_TIME'] % 100
).astype('int16')
print(f"\n=== dep_minutes_since_midnight ===")
print(flights['dep_minutes_since_midnight'].describe().round(2))

flights['is_weekend'] = flights['DAY_OF_WEEK'].isin([6, 7]).astype('int8')
print(f"\n=== is_weekend delay rate ===")
print(flights.groupby('is_weekend')['ARR_DEL15'].mean().round(4) * 100)

dest_hour_daily = (
    flights.groupby(['DEST', 'ARR_HOUR', 'FL_DATE'])['ARR_DEL15']
    .mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)
dest_hour_daily = dest_hour_daily.sort_values(['DEST', 'ARR_HOUR', 'FL_DATE'])
dest_hour_daily['dest_hour_delay_rate_30d'] = (
    dest_hour_daily.groupby(['DEST', 'ARR_HOUR'])['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)
flights = flights.merge(
    dest_hour_daily[['DEST', 'ARR_HOUR', 'FL_DATE', 'dest_hour_delay_rate_30d']],
    on=['DEST', 'ARR_HOUR', 'FL_DATE'], how='left'
)
print(f"\n=== dest_hour_delay_rate_30d ===")
print(f"Nulls: {flights['dest_hour_delay_rate_30d'].isna().sum():,} ({flights['dest_hour_delay_rate_30d'].isna().mean()*100:.1f}%)")

print(f"\nFinal shape: {flights.shape}")

=== tail_flight_num_today ===
tail_flight_num_today
0     4516145
1     4172702
2     3561724
3     2832970
4     1714346
5     1005141
6      283135
7      111191
8       16530
9        8232
10       4715
11        889
12         75
13          1
Name: count, dtype: int64

Delay rate by flight number today:
tail_flight_num_today
0     14.20
1     19.06
2     22.45
3     26.08
4     28.28
5     28.58
6     31.35
7     28.39
8     26.34
9     20.55
10    16.71
11    21.93
Name: ARR_DEL15, dtype: float64

=== dest_hourly_flight_count ===
Nulls: 0
count    18227796.00
mean           22.52
std            20.26
min             1.00
25%             7.00
50%            17.00
75%            31.00
max           113.00
Name: dest_hourly_flight_count, dtype: float64

=== dep_minutes_since_midnight ===
count    18227796.00
mean          807.33
std           298.07
min             1.00
25%           545.00
50%           800.00
75%          1058.00
max          1440.00
Name: dep_minutes_since_midnig

In [ ]:
flights.drop(columns=['dep_minutes_since_midnight', 'is_weekend'], inplace=True)

print("dest_hourly_flight_count vs hourly_flight_count correlation:")
print(flights[['dest_hourly_flight_count', 'hourly_flight_count']].corr().round(4))

print(f"\nShape: {flights.shape}")

flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

dest_hourly_flight_count vs hourly_flight_count correlation:
                          dest_hourly_flight_count  hourly_flight_count
dest_hourly_flight_count                    1.0000              -0.3036
hourly_flight_count                        -0.3036               1.0000

Shape: (18227796, 109)
Saved. Size: 1236.5 MB | Shape: (18227796, 109)


In [ ]:
origin_hourly_arr_delay = flights.groupby(['ORIGIN', 'FL_DATE', 'ARR_HOUR']).agg(
    avg_arr_delay=('ARR_DELAY', 'mean'),
    delayed_pct=('ARR_DEL15', 'mean')
).reset_index()

print("=== Airport hourly arrival delay pattern (sample ATL) ===")
atl = origin_hourly_arr_delay[origin_hourly_arr_delay['ORIGIN'] == 'ATL']
print(atl.groupby('ARR_HOUR')[['avg_arr_delay', 'delayed_pct']].mean().round(2))

top_hubs = ['ATL', 'DEN', 'DFW', 'ORD', 'LAX']
hub_flights = flights[flights['ORIGIN'].isin(top_hubs)]
print(f"\n=== Hub delay rates ===")
print(hub_flights.groupby('ORIGIN')['ARR_DEL15'].mean().round(4) * 100)

late_cascade = flights[(flights['DEP_HOUR'] >= 21) & (flights['cascade_score'] > 0)]
late_no_cascade = flights[(flights['DEP_HOUR'] >= 21) & (flights['cascade_score'] == 0)]
early_cascade = flights[(flights['DEP_HOUR'] <= 9) & (flights['cascade_score'] > 0)]
print(f"\n=== Late night + cascade interaction ===")
print(f"Late night (9PM+) WITH cascade: {late_cascade['ARR_DEL15'].mean()*100:.1f}% ({len(late_cascade):,} flights)")
print(f"Late night (9PM+) NO cascade:   {late_no_cascade['ARR_DEL15'].mean()*100:.1f}% ({len(late_no_cascade):,} flights)")
print(f"Early morning WITH cascade:      {early_cascade['ARR_DEL15'].mean()*100:.1f}% ({len(early_cascade):,} flights)")

print(f"\n=== route_delay_rate_30d unique signal ===")
print(f"Correlation with origin_delay_rate_30d: {flights['route_delay_rate_30d'].corr(flights['origin_delay_rate_30d']):.4f}")
print(f"Correlation with dest_delay_rate_30d:   {flights['route_delay_rate_30d'].corr(flights['dest_delay_rate_30d']):.4f}")

=== Airport hourly arrival delay pattern (sample ATL) ===
          avg_arr_delay  delayed_pct
ARR_HOUR                            
0                 16.27         0.28
1                 18.89         0.33
6                 -0.01         0.11
7                  0.28         0.11
8                  0.22         0.12
9                  1.11         0.13
10                 1.50         0.14
11                 3.24         0.16
12                 3.64         0.16
13                 4.49         0.17
14                 5.69         0.19
15                 6.50         0.20
16                 8.13         0.22
17                10.87         0.25
18                13.97         0.27
19                14.99         0.28
20                15.64         0.29
21                15.55         0.29
22                14.01         0.28
23                13.33         0.26

=== Hub delay rates ===
ORIGIN
ATL    21.22
DEN    25.28
DFW    26.74
LAX    18.37
ORD    23.75
Name: ARR_DEL15, dtype: float64

## Inbound Arrival Delay (3h Lookback) — Origin
Average arrival delay at origin airport over the prior 3 hours. Strongest real-time signal for departure delays.

In [ ]:
flights['ARR_DATETIME'] = pd.to_datetime(
    flights['FL_DATE'].astype(str)
) + pd.to_timedelta(
    (flights['CRS_ARR_TIME'] // 100).astype(int) * 60 + 
    (flights['CRS_ARR_TIME'] % 100).astype(int),
    unit='m'
)

print("Building inbound arrival delay... this will take a few minutes.")
import time
start = time.time()

arrivals = flights[['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy()
arrivals = arrivals.rename(columns={'DEST': 'AIRPORT'})

arr_hourly = arrivals.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index()
arr_hourly = arr_hourly.rename(columns={'ARR_DELAY': 'hourly_arr_delay'})

arr_hourly = arr_hourly.sort_values(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])

results = []
for hour_offset in range(3):
    temp = arr_hourly.copy()
    temp['DEP_HOUR'] = temp['ARR_HOUR'] + hour_offset + 1 
    temp = temp[temp['DEP_HOUR'] <= 23] 
    results.append(temp[['AIRPORT', 'FL_DATE', 'DEP_HOUR', 'hourly_arr_delay']])

combined = pd.concat(results)
inbound_3h = combined.groupby(['AIRPORT', 'FL_DATE', 'DEP_HOUR'])['hourly_arr_delay'].mean().reset_index()
inbound_3h = inbound_3h.rename(columns={
    'AIRPORT': 'ORIGIN',
    'hourly_arr_delay': 'inbound_arr_delay_3h'
})

flights = flights.merge(inbound_3h, on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')

elapsed = time.time() - start
print(f"Done in {elapsed:.1f} seconds")
print(f"Shape: {flights.shape}")
print(f"Nulls: {flights['inbound_arr_delay_3h'].isna().sum():,} ({flights['inbound_arr_delay_3h'].isna().mean()*100:.1f}%)")
print(f"\nDelay rate by inbound_arr_delay_3h quintile:")
non_null = flights[flights['inbound_arr_delay_3h'].notna()]
quintiles = pd.qcut(non_null['inbound_arr_delay_3h'], q=5, duplicates='drop')
print(non_null.groupby(quintiles, observed=True)['ARR_DEL15'].mean().round(4) * 100)

Building inbound arrival delay... this will take a few minutes.
Done in 19.9 seconds
Shape: (18227796, 111)
Nulls: 1,830,425 (10.0%)

Delay rate by inbound_arr_delay_3h quintile:
inbound_arr_delay_3h
(-78.001, -8.417]    10.75
(-8.417, -3.053]     14.67
(-3.053, 2.852]      19.29
(2.852, 12.645]      25.72
(12.645, 2679.0]     41.45
Name: ARR_DEL15, dtype: float64


In [7]:
flights.to_parquet("dataset/merged_flights_fe.parquet",
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")


Saved. Size: 1314.6 MB | Shape: (18227796, 111)


## Inbound Arrival Delay (3h Lookback) — Destination
Average arrival delay arriving at destination over prior 3 hours.

In [ ]:
dest_arrivals = flights[['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy()
dest_arrivals = dest_arrivals.rename(columns={'DEST': 'AIRPORT'})

dest_arr_hourly = dest_arrivals.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index()
dest_arr_hourly = dest_arr_hourly.rename(columns={'ARR_DELAY': 'hourly_arr_delay'})

results = []
for hour_offset in range(3):
    temp = dest_arr_hourly.copy()
    temp['ARR_HOUR_lookup'] = temp['ARR_HOUR'] + hour_offset + 1
    temp = temp[temp['ARR_HOUR_lookup'] <= 23]
    results.append(temp[['AIRPORT', 'FL_DATE', 'ARR_HOUR_lookup', 'hourly_arr_delay']])

combined = pd.concat(results)
dest_inbound = combined.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR_lookup'])['hourly_arr_delay'].mean().reset_index()
dest_inbound = dest_inbound.rename(columns={
    'AIRPORT': 'DEST',
    'ARR_HOUR_lookup': 'ARR_HOUR',
    'hourly_arr_delay': 'dest_inbound_arr_delay_3h'
})

flights = flights.merge(dest_inbound, on=['DEST', 'FL_DATE', 'ARR_HOUR'], how='left')

print("=== dest_inbound_arr_delay_3h ===")
print(f"Nulls: {flights['dest_inbound_arr_delay_3h'].isna().sum():,} ({flights['dest_inbound_arr_delay_3h'].isna().mean()*100:.1f}%)")
quintiles = pd.qcut(flights['dest_inbound_arr_delay_3h'].dropna(), q=5, duplicates='drop')
print(flights.groupby(quintiles, observed=True)['ARR_DEL15'].mean().round(4) * 100)

carrier_daily = flights.groupby(['OP_UNIQUE_CARRIER', 'FL_DATE'])['ARR_DEL15'].mean().reset_index()
carrier_daily = carrier_daily.rename(columns={'ARR_DEL15': 'daily_delay_rate'})
carrier_daily = carrier_daily.sort_values(['OP_UNIQUE_CARRIER', 'FL_DATE'])

carrier_daily['airline_delay_rate_7d'] = (
    carrier_daily.groupby('OP_UNIQUE_CARRIER')['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=3).mean())
)

flights = flights.merge(
    carrier_daily[['OP_UNIQUE_CARRIER', 'FL_DATE', 'airline_delay_rate_7d']],
    on=['OP_UNIQUE_CARRIER', 'FL_DATE'], how='left'
)

print(f"\n=== airline_delay_rate_7d ===")
print(f"Nulls: {flights['airline_delay_rate_7d'].isna().sum():,} ({flights['airline_delay_rate_7d'].isna().mean()*100:.1f}%)")
print(f"Correlation with 30d: {flights['airline_delay_rate_7d'].corr(flights['airline_delay_rate_30d']):.4f}")

origin_daily = flights.groupby(['ORIGIN', 'FL_DATE'])['ARR_DEL15'].mean().reset_index()
origin_daily = origin_daily.rename(columns={'ARR_DEL15': 'daily_delay_rate'})
origin_daily = origin_daily.sort_values(['ORIGIN', 'FL_DATE'])

origin_daily['origin_delay_rate_7d'] = (
    origin_daily.groupby('ORIGIN')['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=3).mean())
)

flights = flights.merge(
    origin_daily[['ORIGIN', 'FL_DATE', 'origin_delay_rate_7d']],
    on=['ORIGIN', 'FL_DATE'], how='left'
)

print(f"\n=== origin_delay_rate_7d ===")
print(f"Nulls: {flights['origin_delay_rate_7d'].isna().sum():,} ({flights['origin_delay_rate_7d'].isna().mean()*100:.1f}%)")
print(f"Correlation with 30d: {flights['origin_delay_rate_7d'].corr(flights['origin_delay_rate_30d']):.4f}")

print(f"\nFinal shape: {flights.shape}")

=== dest_inbound_arr_delay_3h ===
Nulls: 966,910 (5.3%)
dest_inbound_arr_delay_3h
(-78.001, -8.0]     10.63
(-8.0, -2.5]        13.66
(-2.5, 3.7]         17.93
(3.7, 14.156]       24.33
(14.156, 2679.0]    39.96
Name: ARR_DEL15, dtype: float64

=== airline_delay_rate_7d ===
Nulls: 49,432 (0.3%)
Correlation with 30d: 0.7937

=== origin_delay_rate_7d ===
Nulls: 49,625 (0.3%)
Correlation with 30d: 0.7493

Final shape: (18227796, 114)


## Previous Tail Arrival Delay
Arrival delay of the previous flight of this aircraft. Set to NaN if gap exceeds 6 hours (overnight reset).

In [ ]:
import numpy as np

flights_sorted = flights[['TAIL_NUM', 'DEP_DATETIME', 'ARR_DELAY']].copy()
flights_sorted = flights_sorted.sort_values(['TAIL_NUM', 'DEP_DATETIME'])

flights_sorted['prev_tail_arr_delay'] = flights_sorted.groupby('TAIL_NUM')['ARR_DELAY'].shift(1)

flights_sorted['prev_dep'] = flights_sorted.groupby('TAIL_NUM')['DEP_DATETIME'].shift(1)
gap_hours = (flights_sorted['DEP_DATETIME'] - flights_sorted['prev_dep']).dt.total_seconds() / 3600
flights_sorted.loc[gap_hours > 6, 'prev_tail_arr_delay'] = np.nan

flights['prev_tail_arr_delay'] = flights_sorted['prev_tail_arr_delay'].values

print("=== prev_tail_arr_delay ===")
print(f"Nulls: {flights['prev_tail_arr_delay'].isna().sum():,} ({flights['prev_tail_arr_delay'].isna().mean()*100:.1f}%)")
print(flights['prev_tail_arr_delay'].describe().round(2))

print("\nDelay rate by prev_tail_arr_delay bucket:")
buckets = pd.cut(flights['prev_tail_arr_delay'], bins=[-100, -10, 0, 15, 30, 60, 120, 500])
print(flights.groupby(buckets, observed=True)['ARR_DEL15'].mean().round(4) * 100)

=== prev_tail_arr_delay ===
Nulls: 5,705,616 (31.3%)
count    12522180.00
mean            3.50
std            44.17
min           -98.00
25%           -15.00
50%            -7.00
75%             7.00
max          4405.00
Name: prev_tail_arr_delay, dtype: float64

Delay rate by prev_tail_arr_delay bucket:
prev_tail_arr_delay
(-100, -10]    21.30
(-10, 0]       21.48
(0, 15]        21.42
(15, 30]       21.49
(30, 60]       21.38
(60, 120]      21.39
(120, 500]     21.12
Name: ARR_DEL15, dtype: float64


In [ ]:
flights.drop(columns=['prev_tail_arr_delay'], inplace=True)

flights_sorted = flights[['TAIL_NUM', 'DEP_DATETIME', 'ARR_DELAY']].copy()
flights_sorted = flights_sorted.sort_values(['TAIL_NUM', 'DEP_DATETIME'])

flights_sorted['prev_tail_arr_delay'] = flights_sorted.groupby('TAIL_NUM')['ARR_DELAY'].shift(1)
flights_sorted['prev_dep'] = flights_sorted.groupby('TAIL_NUM')['DEP_DATETIME'].shift(1)
gap_hours = (flights_sorted['DEP_DATETIME'] - flights_sorted['prev_dep']).dt.total_seconds() / 3600
flights_sorted.loc[gap_hours > 6, 'prev_tail_arr_delay'] = np.nan

flights = flights.join(flights_sorted[['prev_tail_arr_delay']], how='left')

print("=== prev_tail_arr_delay (index-aligned) ===")
print(f"Nulls: {flights['prev_tail_arr_delay'].isna().sum():,} ({flights['prev_tail_arr_delay'].isna().mean()*100:.1f}%)")

print("\nDelay rate by prev_tail_arr_delay bucket:")
buckets = pd.cut(flights['prev_tail_arr_delay'], bins=[-100, -10, 0, 15, 30, 60, 120, 500])
print(flights.groupby(buckets, observed=True)['ARR_DEL15'].mean().round(4) * 100)

print("\nN476HA Jan 12 2024:")
print(flights[
    (flights['TAIL_NUM'] == 'N476HA') & (flights['FL_DATE'] == '2024-01-12')
][['CRS_DEP_TIME', 'ARR_DELAY', 'prev_tail_arr_delay', 'cascade_score']].sort_values('CRS_DEP_TIME').to_string())

=== prev_tail_arr_delay (index-aligned) ===
Nulls: 5,705,616 (31.3%)

Delay rate by prev_tail_arr_delay bucket:
prev_tail_arr_delay
(-100, -10]    10.76
(-10, 0]       12.73
(0, 15]        21.14
(15, 30]       44.26
(30, 60]       75.16
(60, 120]      89.73
(120, 500]     86.98
Name: ARR_DEL15, dtype: float64

N476HA Jan 12 2024:
          CRS_DEP_TIME  ARR_DELAY  prev_tail_arr_delay  cascade_score
13383190           536        4.0                  NaN              0
13383192           645       -1.0                  4.0              0
13383194           800       -3.0                 -1.0              0
13383193           917        1.0                 -3.0              0
13383195          1041        2.0                  1.0              0
13383197          1209        3.0                  2.0              0
13383196          1320        0.0                  3.0              0
13383199          1440       15.0                  0.0              0
13383198          1550       20.0     

## National Hub Delay (2h Lookback)
Average arrival delay at top 10 hub airports over prior 2 hours. Captures nationwide system-level congestion.

In [ ]:
top10_hubs = ['ATL', 'DEN', 'DFW', 'ORD', 'LAX', 'CLT', 'MCO', 'LAS', 'PHX', 'MIA']

hub_arrivals = flights[flights['DEST'].isin(top10_hubs)][['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy()
hub_hourly = hub_arrivals.groupby(['FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index()
hub_hourly = hub_hourly.rename(columns={'ARR_DELAY': 'hub_hourly_delay'})
hub_hourly = hub_hourly.sort_values(['FL_DATE', 'ARR_HOUR'])

results = []
for offset in range(2):
    temp = hub_hourly.copy()
    temp['DEP_HOUR'] = temp['ARR_HOUR'] + offset + 1
    temp = temp[temp['DEP_HOUR'] <= 23]
    results.append(temp[['FL_DATE', 'DEP_HOUR', 'hub_hourly_delay']])

combined = pd.concat(results)
hub_fever = combined.groupby(['FL_DATE', 'DEP_HOUR'])['hub_hourly_delay'].mean().reset_index()
hub_fever = hub_fever.rename(columns={'hub_hourly_delay': 'national_hub_delay_2h'})

flights = flights.merge(hub_fever, on=['FL_DATE', 'DEP_HOUR'], how='left')

print("=== national_hub_delay_2h ===")
print(f"Nulls: {flights['national_hub_delay_2h'].isna().sum():,} ({flights['national_hub_delay_2h'].isna().mean()*100:.1f}%)")

print("\nDelay rate by national hub delay:")
buckets = pd.cut(flights['national_hub_delay_2h'], bins=[-50, -5, 0, 5, 10, 20, 50, 500])
print(flights.groupby(buckets, observed=True)['ARR_DEL15'].mean().round(4) * 100)

=== national_hub_delay_2h ===
Nulls: 30,864 (0.2%)

Delay rate by national hub delay:
national_hub_delay_2h
(-50, -5]    10.40
(-5, 0]      14.45
(0, 5]       19.10
(5, 10]      23.40
(10, 20]     28.95
(20, 50]     36.95
(50, 500]    38.97
Name: ARR_DEL15, dtype: float64


In [8]:
flights.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM',
       ...
       'tail_flight_num_today', 'dest_hourly_flight_count',
       'dest_hour_delay_rate_30d', 'ARR_DATETIME', 'inbound_arr_delay_3h',
       'dest_inbound_arr_delay_3h', 'airline_delay_rate_7d',
       'origin_delay_rate_7d', 'prev_tail_arr_delay', 'national_hub_delay_2h'],
      dtype='object', length=116)

In [9]:
print("Correlations between new features:")
new_feats = ['inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h', 
             'prev_tail_arr_delay', 'national_hub_delay_2h']
print(flights[new_feats].corr().round(4))

Correlations between new features:
                           inbound_arr_delay_3h  dest_inbound_arr_delay_3h  \
inbound_arr_delay_3h                     1.0000                     0.2309   
dest_inbound_arr_delay_3h                0.2309                     1.0000   
prev_tail_arr_delay                      0.3387                     0.1699   
national_hub_delay_2h                    0.4224                     0.3797   

                           prev_tail_arr_delay  national_hub_delay_2h  
inbound_arr_delay_3h                    0.3387                 0.4224  
dest_inbound_arr_delay_3h               0.1699                 0.3797  
prev_tail_arr_delay                     1.0000                 0.2122  
national_hub_delay_2h                   0.2122                 1.0000  


## Save Dataset V3

In [10]:
flights.to_parquet("dataset/merged_flights_fe.parquet",
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")


Saved. Size: 1419.6 MB | Shape: (18227796, 116)
